In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_R_K_Puram_Delhi_DPCC_2023.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,258.0,197.0,170.0,92.0,110.0,95.0,65.0,76.0,156.0,164.0,378.0,398.0
1,2,393.0,207.0,264.0,112.0,99.0,116.0,58.0,85.0,137.0,158.0,410.0,385.0
2,3,402.0,201.0,170.0,205.0,117.0,166.0,143.0,77.0,141.0,168.0,498.0,342.0
3,4,373.0,245.0,113.0,129.0,210.0,214.0,191.0,154.0,144.0,179.0,430.0,344.0
4,5,371.0,249.0,121.0,174.0,203.0,191.0,104.0,91.0,230.0,192.0,485.0,324.0
5,6,442.0,251.0,148.0,141.0,258.0,161.0,83.0,115.0,113.0,215.0,449.0,315.0
6,7,397.0,285.0,198.0,137.0,168.0,293.0,67.0,104.0,116.0,204.0,430.0,319.0
7,8,404.0,121.0,244.0,209.0,167.0,168.0,62.0,118.0,83.0,153.0,443.0,319.0
8,9,472.0,223.0,119.0,238.0,226.0,132.0,72.0,130.0,46.0,165.0,466.0,335.0
9,10,436.0,192.0,213.0,222.0,237.0,121.0,NaN,134.0,46.0,NaN,289.0,346.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      35 non-null     float64
 4   April      34 non-null     float64
 5   May        36 non-null     float64
 6   June       34 non-null     float64
 7   July       25 non-null     float64
 8   August     33 non-null     float64
 9   September  35 non-null     float64
 10  October    34 non-null     float64
 11  November   34 non-null     float64
 12  December   32 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [10]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [11]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [12]:
# Convert all columns except 'Day' to numeric values
for col in df.columns:
    if col != 'Day':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values with column mean
df_filled = df.fillna(df.mean(numeric_only=True))

In [13]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [15]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,258.0,197.0,170.0,92.0,110.0,95.0,65.00,76.0,156.000000,164.0,378.0,398.0
1,2,393.0,207.0,264.0,112.0,99.0,116.0,58.00,85.0,137.000000,158.0,410.0,385.0
2,3,402.0,201.0,170.0,205.0,117.0,166.0,70.52,77.0,141.000000,168.0,498.0,342.0
3,4,373.0,245.0,113.0,129.0,210.0,214.0,70.52,154.0,144.000000,179.0,430.0,344.0
4,5,371.0,249.0,121.0,174.0,203.0,191.0,70.52,91.0,108.285714,192.0,485.0,324.0
